# Week 7 — Recurrent Neural Networks (RNN) for Time Series Forecasting (Keras)

# Sources:
1. https://cs231n.github.io/rnn/
2. https://visionbook.mit.edu/recurrent_neural_nets.html

## Learning goals
- what an RNN is and why it is useful for sequences;
- the ideas of *unrolling in time* and *Backpropagation Through Time (BPTT)*;
- the ideas behind the Long Short-Term Memory (LSTM), a variant of the RNN;
- how to prepare a time-series dataset with *sliding windows*;
- how to train an LSTM in Keras for **one-step** forecasting and compare it with a baseline.

---

## 1) Why sequences are different
Classical supervised learning often assumes that observations are independent (i.i.d.). In a time series, the value at time $t$ depends on earlier values: $x_t$ is correlated with $x_{t-1}, x_{t-2}, \dots$.

A common workaround is to add manually engineered lag features, but recurrent models learn temporal dependencies automatically.

---

## 2) Vanilla RNN — key equations
An RNN maintains a hidden state $h_t$ (its memory). For a sequence $x_1, x_2, \dots, x_T$:

$$h_t = \phi(W_{xh} x_t + W_{hh} h_{t-1} + b)$$
$$\hat{y}_t = g(W_{hy} h_t + c)$$

<img src="https://visionbook.mit.edu/figures/recurrent_neural_nets/rnn_unrolled.png">

<img src="https://cs231n.github.io/assets/rnn/vanilla_rnn_mformula_1.png">

<img src="https://cs231n.github.io/assets/rnn/vanilla_rnn_mformula_2.png">

Key points:
- the matrices $W_{xh}, W_{hh}, W_{hy}$ are **shared across all time steps**;
- the same computation is repeated for every $t$ (*unrolling*);
- training uses **BPTT** (Backpropagation Through Time, i.e. backpropagation over the unrolled computational graph).

In slightly different notation, we unroll the dependency graph in time for the backpropagation algorithm:

$$
\begin{aligned}
h_t &= \sigma_1\!\left(W h_{t-1} + U x_{\text{in}}[t] + b\right) \\
x_{\text{out}}[t] &= \sigma_2\!\left(V h_t + c\right)
\end{aligned}
$$

<img src="https://visionbook.mit.edu/figures/recurrent_neural_nets/bptt1.png">

The total loss $J$ can also be computed as the sum of the discrepancies $\mathcal{L}$ at each predicted output. The red arrows in the figure below point opposite to the forward flow of information (black arrows); they describe backpropagation when computing gradients.

<img src="https://visionbook.mit.edu/figures/recurrent_neural_nets/bptt_J.png">

### Vanishing / exploding gradients (intuition)
Because the model is equivalent to a very deep network "in time", gradients can vanish or explode.
Practical remedies:
- gated cells (LSTM / GRU);
- gradient clipping.

---

## 3) LSTM (Long Short-Term Memory) — why it is better
An LSTM has a *cell state* $c_t$ and gates that control the flow of information. This helps it keep useful signals over longer horizons.

An LSTM controls information through gates — ordinary sigmoid layers that output values in the range $[0,1]$.

---

**Forget gate** $f_t$ — what to forget (for example, if seasonality is no longer relevant, the forget gate ≈ 0)

$$f_t = \sigma(W_{hf}h_{t-1} + W_{xf}x_t + b_f) $$

---

**Input gate** $i_t, g_t$ — what to write

The input gate decides **which new information** to write into the LSTM memory and **how much** of it.

#### 1. Which new information to add (candidate memory)

The candidate new information is computed as:

$$
g_t = \tanh\!\left(W_c x_t + U_c h_{t-1} + b_c\right)
$$

#### 2. How much of it to let in (input gate)

The input gate controls which part of the candidate $g_t$ is written to memory:

$$
i_t = \sigma\!\left(W_i x_t + U_i h_{t-1} + b_i\right)
$$

#### 3. Memory update (cell state update)

The final update of the memory state:

$$
c_t = f_t \odot c_{t-1} + i_t \odot g_t
$$

where:
- $f_t$ — forget gate,
- $\odot$ — element-wise multiplication.

Role:
- the left term → memory from the past;
- the right term → new information.

---

**Output gate** $o_t$ — what to expose

$$
o_t = \sigma\!\left(W_o x_t + U_o h_{t-1} + b_o\right)
$$

The final hidden state is a filtered version of the cell state:

$$
h_t = o_t \odot \tanh\!\left(c_t\right)
$$

---

All formulas as images (the gates are combined into one matrix for computational convenience):

<img src="https://cs231n.github.io/assets/rnn/lstm_mformula_1.png">

<img src="https://cs231n.github.io/assets/rnn/lstm_mformula_2.png">

**Important:** you do not need to memorise the gate formulas. What matters is that:
- there is a separate memory path;
- the gates control what is forgotten, written and exposed.

This is how information flows through an LSTM:

<img src="https://cs231n.github.io/assets/rnn/lstm_highway.png">

---

## 4) Forecasting setup (one-step)

We now work through a practical example of one-step-ahead forecasting on a time series.

The standard sliding-window formulation:
- input: $[x_{t-w}, \dots, x_{t-1}]$ (a window of length $w$);
- target: $x_t$ (the next value).

We compare:
- **Naive baseline**: $\hat{x}_t = x_{t-1}$
- **LSTM model**: a non-linear function that maps the window to the next value.


In [ ]:
#@title Setup
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print('TensorFlow:', tf.__version__)
np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams['figure.figsize'] = (12, 4)


## 5) Synthetic time series
We generate a signal with trend, seasonality and noise:

$$x_t = 0.0008\,t + 2\sin\!\left(\frac{2\pi t}{50}\right) + 0.8\sin\!\left(\frac{2\pi t}{16}\right) + \varepsilon_t, \qquad \varepsilon_t \sim \mathcal{N}(0, 0.3^2)$$

This is enough to demonstrate:
- the role of context (the window);
- a correct time-based split;
- whether an LSTM improves on the baseline.


In [ ]:
#@title Generate a synthetic time series
def make_series(n=4000, noise_std=0.3, seed=42):
    rng = np.random.default_rng(seed)  # local generator: the series is reproducible on its own
    t = np.arange(n, dtype=np.float32)
    trend = 0.0008 * t
    s1 = 2.0 * np.sin(2*np.pi*t/50.0)
    s2 = 0.8 * np.sin(2*np.pi*t/16.0)
    noise = rng.normal(0, noise_std, size=n).astype(np.float32)
    x = trend + s1 + s2 + noise
    return t, x

t, x = make_series()
print('Series length:', len(x), 'min/max:', float(x.min()), float(x.max()))


In [ ]:
#@title Plot the series (first 800 points)
plt.plot(t[:800], x[:800])
plt.title('Synthetic time series (first 800 points)')
plt.xlabel('t')
plt.ylabel('x(t)')
plt.grid(True)
plt.show()


## 6) Windowed dataset
We convert the 1-D series into supervised samples.

- Each input has shape $(w, 1)$, i.e. **(time, features)**.
- Keras recurrent layers expect shape **(batch, time, features)**.

We also use a **time-based split** (without shuffling) to avoid leakage, and we standardise the data using training-period statistics only.

> **Data leakage in time series.**
>
> Leakage occurs when, during training or validation, the model receives information from future time steps.
> In forecasting this usually happens through shuffling, fitting preprocessing (such as normalisation) on the full series, or incorrect windowing.


In [ ]:
#@title Build a windowed dataset (NumPy)
def make_windows(series, window=40):
    X, y = [], []
    for i in range(window, len(series)):
        X.append(series[i-window:i])
        y.append(series[i])
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    return X[..., None], y[..., None]  # add a feature dimension

WINDOW = 40
X_all, y_all = make_windows(x, window=WINDOW)
print('X:', X_all.shape, 'y:', y_all.shape)


In [ ]:
#@title Train/val/test split (time-based) and standardisation
n = len(X_all)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

X_train, y_train = X_all[:train_end], y_all[:train_end]
X_val, y_val = X_all[train_end:val_end], y_all[train_end:val_end]
X_test, y_test = X_all[val_end:], y_all[val_end:]

# Standardise with statistics from the training period only.
# The last training target is series index train_end + WINDOW - 1.
train_points = x[:train_end + WINDOW]
MU, SIGMA = float(train_points.mean()), float(train_points.std())

def scale(a):
    return (a - MU) / SIGMA

def unscale(a):
    return a * SIGMA + MU

X_train, y_train = scale(X_train), scale(y_train)
X_val, y_val = scale(X_val), scale(y_val)
X_test, y_test = scale(X_test), scale(y_test)

print('Train:', X_train.shape, y_train.shape)
print('Val:  ', X_val.shape, y_val.shape)
print('Test: ', X_test.shape, y_test.shape)
print(f'Standardisation (training period): mean={MU:.3f}, std={SIGMA:.3f}')


## 7) Baseline: naive forecast
The simplest baseline predicts that the next value equals the last observed value in the window:

$$\hat{x}_t = x_{t-1}$$

We evaluate MAE and RMSE on the test set, in the original units of the series.

> MAE — Mean Absolute Error, RMSE — Root Mean Squared Error.


In [ ]:
#@title Baseline evaluation (original units)
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

# Map targets and predictions back to the original scale before computing the metrics
y_test_orig = unscale(y_test)
y_pred_baseline = unscale(X_test[:, -1, 0:1])  # the last value of each input window
print('Baseline MAE :', mae(y_test_orig, y_pred_baseline))
print('Baseline RMSE:', rmse(y_test_orig, y_pred_baseline))


## 8) LSTM model (Keras)
We train a small LSTM for one-step forecasting.

- Input shape: $(w, 1)$
- Output: $1$ value
- Loss: MSE, metric: MAE

We add EarlyStopping to prevent overfitting.


In [ ]:
#@title Build the LSTM model
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout

def build_lstm(window):
    model = Sequential([
        Input(shape=(window, 1)),
        LSTM(64),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = build_lstm(WINDOW)
model.summary()


In [ ]:
#@title Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=8, restore_best_weights=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=64,
    shuffle=True,  # reorders the training windows only; time order is kept inside each window
    verbose=1,
    callbacks=callbacks
)


In [ ]:
#@title Learning curves
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss (MSE)')
plt.xlabel('epoch')
plt.ylabel('mse')
plt.grid(True)
plt.legend()
plt.show()


## 9) Test-set evaluation
We compare the baseline and the LSTM on the same test slice, in the original units of the series. The test set is used only once, after training and early stopping are complete.


In [ ]:
#@title Test metrics: baseline vs LSTM
y_pred_lstm = unscale(model.predict(X_test, verbose=0))

print('Baseline MAE :', mae(y_test_orig, y_pred_baseline))
print('LSTM MAE     :', mae(y_test_orig, y_pred_lstm))
print('Baseline RMSE:', rmse(y_test_orig, y_pred_baseline))
print('LSTM RMSE    :', rmse(y_test_orig, y_pred_lstm))


**Interpretation.** Compare the LSTM with the naive baseline on the same test points. Both metrics are in the units of $x(t)$: an MAE or RMSE below the baseline's means the network captures structure beyond the last observed value. RMSE penalises large errors more heavily than MAE, so a large gap between the two suggests occasional large misses. The residual histogram (below) shows whether the errors are centred on zero (no systematic bias) and how widely they are spread.


In [ ]:
#@title Visualise predictions (zoom)
k = 500
plt.plot(y_test_orig[:k], label='true')
plt.plot(y_pred_baseline[:k], label='baseline', alpha=0.7)
plt.plot(y_pred_lstm[:k], label='LSTM', alpha=0.9)
plt.title(f'One-step forecast: true vs predictions (first {k} test points)')
plt.xlabel('time index within test set')
plt.ylabel('x(t)')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
#@title Residual analysis
res_base = (y_test_orig - y_pred_baseline).squeeze()
res_lstm = (y_test_orig - y_pred_lstm).squeeze()

plt.hist(res_base, bins=50, alpha=0.5, label='baseline')
plt.hist(res_lstm, bins=50, alpha=0.5, label='LSTM')
plt.title('Residual distribution on test set')
plt.xlabel('residual = true - prediction')
plt.ylabel('count')
plt.legend()
plt.show()


## 10) (Teaser) Multi-step forecasting and error accumulation
To forecast $H$ steps ahead, there are two common strategies:

1. **Recursive strategy**: predict one step, append the prediction to the window, and repeat $H$ times.
2. **Direct strategy**: train a model that outputs a vector of length $H$ at once.

The recursive approach often accumulates error:

$$\hat{x}_{t+1} = f([x_{t-w+1}, \dots, x_t])$$
$$\hat{x}_{t+2} = f([x_{t-w+2}, \dots, x_t, \hat{x}_{t+1}])$$

This is one of the key challenges in forecasting, and it will be the focus of a more advanced practical task.


In [ ]:
#@title Simple recursive multi-step forecast (illustration)
def recursive_forecast(model, seed_window, horizon=100):
    # Each prediction is fed back into the window (inputs and outputs are on the scaled series)
    window = seed_window.copy().astype(np.float32)
    preds = []
    for _ in range(horizon):
        yhat = model.predict(window[None, ..., None], verbose=0)[0, 0]
        preds.append(yhat)
        window = np.concatenate([window[1:], [yhat]])
    return np.array(preds, dtype=np.float32)

HORIZON = 199  # the last 200 points of the series are held out for this illustration
seed = scale(x[-(WINDOW+200):-200])  # scaled input window
future_true = x[-200:-200+HORIZON]   # original units, for comparison
future_pred = unscale(recursive_forecast(model, seed, horizon=HORIZON))

plt.plot(future_true, label=f'true ({HORIZON} steps)')
plt.plot(future_pred, label='recursive LSTM', alpha=0.9)
plt.title(f'Recursive multi-step forecast ({HORIZON} steps)')
plt.xlabel('horizon step')
plt.ylabel('x(t)')
plt.grid(True)
plt.legend()
plt.show()


> Even an LSTM does not "solve" error accumulation. This is a fundamental limitation of autoregressive forecasting.


## 11) Summary
- RNNs process sequences through the hidden state $h_t$.
- Training uses BPTT; long sequences lead to vanishing or exploding gradients.
- LSTMs mitigate this problem through gating and an explicit memory.
- The typical forecasting formulation is *window → next value*.
- It is important to compare against a baseline and to use a time-based split without leakage. Fit any preprocessing (such as standardisation) on the training period only.

### Optional exercises
1. Change the window length $w$ (for example, $w$ = 10, 20, 40, 80). Set `WINDOW`, re-run from cell 6 onwards, and compare the MAE and RMSE with the baseline.
2. Increase `HORIZON` in the recursive forecast and observe how the prediction drifts away from the true series.
3. Replace the LSTM with a GRU (Gated Recurrent Unit) and compare the results. A GRU is a simplified variant of the LSTM; more details are available in [this article](https://towardsdatascience.com/gru-recurrent-neural-networks-a-smart-way-to-predict-sequences-in-python-80864e4fe9f6/).
